# Chessboard Pattern with Faces

This notebook demonstrates how to create a chessboard pattern using topologic_fast.
We will:
1. Create an 8x8 grid of square faces
2. Assign alternating colors (black/white) to each square
3. Combine them into a Shell
4. Visualize the result using Plotly

**Adapted from topologicpy Chessboard example.**

Note: topologic_fast does not currently support Dictionary operations for storing metadata on topologies.
We handle colors separately in Python for visualization.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
print("Libraries imported successfully.")

## 1. Create the Chessboard Squares

We create 64 square faces (8x8 grid) with alternating colors.
Each square is 1 unit x 1 unit, positioned at integer coordinates.

In [ ]:
squares = []
colors = []  # Store colors for each square

for i in range(8):
    for j in range(8):
        # Create a vertex at position (i, j, 0)
        # Note: topologic_fast Face.Rectangle creates a rectangle centered at origin
        # We need to translate it to the correct position
        
        # Create a 1x1 rectangle face at position (i+0.5, j+0.5, 0)
        # The face is created centered at origin, then we translate
        center_x = i + 0.5
        center_y = j + 0.5
        
        # Create vertices for the square
        v0 = tf.Vertex.ByCoordinates(i, j, 0)
        v1 = tf.Vertex.ByCoordinates(i + 1, j, 0)
        v2 = tf.Vertex.ByCoordinates(i + 1, j + 1, 0)
        v3 = tf.Vertex.ByCoordinates(i, j + 1, 0)
        
        # Create edges using ByStartVertexEndVertex
        e0 = tf.Edge.ByStartVertexEndVertex(v0, v1)
        e1 = tf.Edge.ByStartVertexEndVertex(v1, v2)
        e2 = tf.Edge.ByStartVertexEndVertex(v2, v3)
        e3 = tf.Edge.ByStartVertexEndVertex(v3, v0)
        
        # Create wire and face
        wire = tf.Wire.ByEdges([e0, e1, e2, e3])
        sq = tf.Face.ByWire(wire)
        
        # Determine color (alternating pattern)
        if i % 2 == j % 2:
            color = "#EEEEEE"  # Light squares
        else:
            color = "#000000"  # Dark squares
        
        squares.append(sq)
        colors.append(color)

print(f"Created {len(squares)} squares for the chessboard.")

## 2. Combine Squares into a Shell

We combine all the square faces into a single Shell topology.
A Shell is a connected set of faces sharing edges.

In [ ]:
# Note: Shell.ByFaces may not preserve individual face boundaries in all cases
# For visualization, we'll work with individual squares

shell = tf.Shell.ByFaces(squares)

print(f"Created Shell with:")
print(f"  - Number of faces: {shell.NumFaces()}")
print(f"  - Total area: {shell.Area():.2f} square units")

## 3. Visualize the Chessboard (2D View)

We use Plotly to create a 2D visualization of the chessboard with proper coloring.

In [ ]:
def visualize_chessboard_2d(squares, colors):
    """Create a 2D visualization of the chessboard."""
    fig = go.Figure()
    
    for i, (sq, color) in enumerate(zip(squares, colors)):
        # Get vertices of the face
        vertices = sq.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        # Close the polygon
        x = [c[0] for c in coords] + [coords[0][0]]
        y = [c[1] for c in coords] + [coords[0][1]]
        
        row = i // 8
        col = i % 8
        
        fig.add_trace(go.Scatter(
            x=x, y=y,
            fill='toself',
            fillcolor=color,
            line=dict(color='gray', width=1),
            mode='lines',
            hoverinfo='text',
            text=f"Square ({chr(97+col)}{row+1})",
            showlegend=False
        ))
    
    # Add board labels (a-h, 1-8)
    letters = 'abcdefgh'
    for i, letter in enumerate(letters):
        fig.add_annotation(
            x=i + 0.5, y=-0.3,
            text=letter,
            showarrow=False,
            font=dict(size=14, color='black')
        )
    for i in range(8):
        fig.add_annotation(
            x=-0.3, y=i + 0.5,
            text=str(i + 1),
            showarrow=False,
            font=dict(size=14, color='black')
        )
    
    fig.update_layout(
        title='Chessboard Pattern (topologic_fast)',
        xaxis=dict(
            title='',
            scaleanchor='y',
            scaleratio=1,
            range=[-0.5, 8.5],
            showticklabels=False
        ),
        yaxis=dict(
            title='',
            range=[-0.5, 8.5],
            showticklabels=False
        ),
        width=600,
        height=600,
        plot_bgcolor='white'
    )
    
    return fig

fig_2d = visualize_chessboard_2d(squares, colors)
fig_2d.show()

## 4. Visualize the Chessboard (3D View)

We can also create a 3D visualization to see the board from different angles.

In [ ]:
def visualize_chessboard_3d(squares, colors):
    """Create a 3D visualization of the chessboard."""
    fig = go.Figure()
    
    for sq, color in zip(squares, colors):
        vertices = sq.Vertices()
        coords = [v.Coordinates() for v in vertices]
        
        x = [c[0] for c in coords]
        y = [c[1] for c in coords]
        z = [c[2] for c in coords]
        
        # Create a mesh for the square
        fig.add_trace(go.Mesh3d(
            x=x, y=y, z=z,
            color=color,
            opacity=1.0,
            alphahull=0,
            showlegend=False,
            hoverinfo='skip'
        ))
        
        # Add edges for visibility
        for k in range(len(coords)):
            p1 = coords[k]
            p2 = coords[(k + 1) % len(coords)]
            fig.add_trace(go.Scatter3d(
                x=[p1[0], p2[0]],
                y=[p1[1], p2[1]],
                z=[p1[2], p2[2]],
                mode='lines',
                line=dict(color='gray', width=2),
                showlegend=False,
                hoverinfo='skip'
            ))
    
    fig.update_layout(
        title='3D Chessboard (topologic_fast)',
        scene=dict(
            aspectmode='data',
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0))
        ),
        width=700,
        height=600
    )
    
    return fig

fig_3d = visualize_chessboard_3d(squares, colors)
fig_3d.show()

## 5. Get Square Centroids

In topologicpy, dictionaries are used to store metadata on topologies.
topologic_fast does not yet support Dictionary operations, so we compute
centroids manually for each square.

In [ ]:
# NOTE: Dictionary operations are not available in topologic_fast.
# In topologicpy, you would use:
#   d = Dictionary.ByKeyValue("color", color)
#   sq = Topology.SetDictionary(sq, d)
#   centroid = Topology.Centroid(sq)
# 
# Instead, we compute centroids manually:

def compute_centroid(face):
    """Compute the centroid of a face."""
    vertices = face.Vertices()
    coords = [v.Coordinates() for v in vertices]
    
    cx = sum(c[0] for c in coords) / len(coords)
    cy = sum(c[1] for c in coords) / len(coords)
    cz = sum(c[2] for c in coords) / len(coords)
    
    return (cx, cy, cz)

centroids = [compute_centroid(sq) for sq in squares]

# Display some centroids with their colors
print("Sample square centroids with colors:")
print("-" * 50)
for i in [0, 1, 8, 9, 27, 63]:
    row = i // 8
    col = i % 8
    print(f"  Square {chr(97+col)}{row+1}: centroid = {centroids[i]}, color = {colors[i]}")

## 6. Create Adjacency Graph (Optional)

We can create a graph showing which squares are adjacent to each other.

Note: Graph operations from shell topology may not be fully supported in topologic_fast.
We demonstrate the concept with manual adjacency detection.

In [ ]:
# NOTE: In topologicpy, you would use:
#   g = Graph.ByTopology(shell)
#   vertices = Graph.Vertices(g)
#
# topologic_fast may not support Graph.ByTopology for shells in the same way.
# We create a simple adjacency representation manually:

def get_adjacent_squares(index):
    """Get indices of squares adjacent to the given square."""
    row = index // 8
    col = index % 8
    
    adjacent = []
    
    # Up
    if row < 7:
        adjacent.append((row + 1) * 8 + col)
    # Down
    if row > 0:
        adjacent.append((row - 1) * 8 + col)
    # Right
    if col < 7:
        adjacent.append(row * 8 + col + 1)
    # Left
    if col > 0:
        adjacent.append(row * 8 + col - 1)
    
    return adjacent

# Build adjacency list
adjacency = {i: get_adjacent_squares(i) for i in range(64)}

# Example: Show adjacencies for corner and center squares
print("Adjacency examples:")
print("-" * 40)

for idx in [0, 27, 35, 63]:
    row = idx // 8
    col = idx % 8
    adj_squares = [f"{chr(97 + (a % 8))}{(a // 8) + 1}" for a in adjacency[idx]]
    print(f"  {chr(97+col)}{row+1} is adjacent to: {', '.join(adj_squares)}")

## 7. Visualize Centroids on Chessboard

In [ ]:
def visualize_with_centroids(squares, colors, centroids):
    """Visualize chessboard with centroids marked."""
    fig = go.Figure()
    
    # Draw squares
    for sq, color in zip(squares, colors):
        vertices = sq.Vertices()
        coords = [v.Coordinates() for v in vertices]
        x = [c[0] for c in coords] + [coords[0][0]]
        y = [c[1] for c in coords] + [coords[0][1]]
        
        fig.add_trace(go.Scatter(
            x=x, y=y,
            fill='toself',
            fillcolor=color,
            line=dict(color='gray', width=1),
            mode='lines',
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw centroids
    cx = [c[0] for c in centroids]
    cy = [c[1] for c in centroids]
    
    # Create hover text with square names
    hover_text = [f"{chr(97 + (i % 8))}{(i // 8) + 1}" for i in range(64)]
    
    fig.add_trace(go.Scatter(
        x=cx, y=cy,
        mode='markers',
        marker=dict(
            size=8,
            color='red',
            line=dict(color='darkred', width=1)
        ),
        hovertext=hover_text,
        hoverinfo='text',
        name='Centroids'
    ))
    
    fig.update_layout(
        title='Chessboard with Square Centroids',
        xaxis=dict(
            scaleanchor='y',
            scaleratio=1,
            range=[-0.5, 8.5],
            showticklabels=False
        ),
        yaxis=dict(
            range=[-0.5, 8.5],
            showticklabels=False
        ),
        width=600,
        height=600,
        plot_bgcolor='white'
    )
    
    return fig

fig_centroids = visualize_with_centroids(squares, colors, centroids)
fig_centroids.show()

## Summary

This notebook demonstrated:

1. **Creating Faces** - Using `tf.Vertex.ByCoordinates()`, `tf.Edge.ByStartVertexEndVertex()`, `tf.Wire.ByEdges()`, and `tf.Face.ByWire()` to create square faces
2. **Building a Shell** - Combining faces using `tf.Shell.ByFaces()`
3. **Plotly Visualization** - 2D and 3D views of the chessboard
4. **Computing Centroids** - Manual centroid computation since Dictionary operations are not available
5. **Adjacency Analysis** - Building a graph of adjacent squares

### Key Differences from topologicpy:

- **No Dictionary support**: Cannot attach metadata (like colors) directly to topologies
- **No Topology.Show()**: Must use Plotly or other visualization libraries manually
- **No Topology.Centroid()**: Computed manually from vertex coordinates
- **Graph.ByTopology()**: May work differently or have limited support for shells

### Applications:

- Board game representations
- Grid-based spatial analysis
- Tile pattern generation
- Adjacency graph analysis